# NAM Tutorial 17 — Skin Sensitisation Defined Approach
### Replacing the Guinea Pig Maximisation Test with the OECD TG 497 DA

**Author:** Himanshu Goel | [himanshugoel.github.io](https://himanshugoel.github.io)

---

> **Regulatory context:** OECD TG 497 (2021, updated 2024) formalises Defined
> Approaches for skin sensitisation that are accepted without animal data under EU REACH,
> US EPA TSCA, and NICNAS (Australia). Three DA methods are implemented here:
> **2o3 DA**, **SENS-IS DA**, and the **ITS-3** integrated testing strategy.

## The Adverse Outcome Pathway (AOP #40)

```
MIE: Covalent protein binding (Cys/Lys peptides)     ← DPRA assay
 │
 KE1: Keratinocyte activation (ARE-Nrf2)              ← KeratinoSens / h-CLAT
 │
 KE2: Dendritic cell maturation (CD54/CD86 expression)← h-CLAT, U-SENS
 │
 AO: Skin sensitisation in humans                    ← LLNA / GPMT (replaced)
```

In [ ]:
!pip install rdkit-pypi scikit-learn pandas numpy matplotlib seaborn openai python-dotenv -q
from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec, matplotlib.patches as mpatches
import seaborn as sns, os, json, warnings
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import roc_auc_score, matthews_corrcoef, confusion_matrix
from openai import OpenAI
from dotenv import load_dotenv
warnings.filterwarnings('ignore')
load_dotenv()
os.makedirs('skin_output', exist_ok=True)
AGENT_OK = bool(os.getenv('OPENAI_API_KEY',''))
print('Imports OK')

---
## Step 1 — Chemical Dataset (40 compounds, known LLNA EC3)

In [ ]:
# 40 chemicals with known skin sensitisation from LLNA / human data
# Sources: ECETOC IATA case studies; EURL-ECVAM validation study
SKIN_DATA = [
    # name, SMILES, sensitiser (1=yes,0=no), potency_class, category
    ('DNCB',      'O=[N+]([O-])c1ccc(Cl)c([N+](=O)[O-])c1', 1,'extreme','classic sensitiser'),
    ('Oxazolone', 'O=C1OC(=O)C=C1',                          1,'extreme','murine LLNA model'),
    ('Cinnamaldehyde','O=C/C=C/c1ccccc1',                     1,'strong','Michael acceptor'),
    ('Eugenol',   'C=CCc1ccc(O)c(OC)c1',                     1,'moderate','fragrance'),
    ('Isoeugenol','C/C=C/c1ccc(O)c(OC)c1',                   1,'strong','fragrance'),
    ('Benzyl benzoate','O=C(OCc1ccccc1)c1ccccc1',              0,'non','fragrance – non-sensitiser'),
    ('Linalool',  'CC(C)=CCCC(C)(O)C=C',                     0,'non','fragrance'),
    ('Geraniol',  'CC(C)=CCC/C(C)=C/CO',                     0,'non','fragrance'),
    ('Salicylaldehyde','OC(=O)c1ccccc1C=O',                   1,'moderate','aldehyde'),
    ('Formaldehyde','C=O',                                     1,'extreme','Schiff base former'),
    ('Glutaraldehyde','O=CCCCC=O',                             1,'extreme','bis-aldehyde'),
    ('MCI',       'CC1(Cl)SC(=O)N(CCCO)C1=O',                 1,'extreme','preservative'),
    ('Kathon-CG', 'CC1=NC(=O)N(CCCO)C1=O',                    1,'extreme','preservative'),
    ('PPD',       'Nc1ccc(N)cc1',                              1,'extreme','hair dye'),
    ('Hexyl cinnamic aldehyde','O=C/C=C/c1ccccc1CCCCCC',       1,'moderate','fragrance'),
    ('Lily aldehyde','O=Cc1ccc(CC(C)(C)C)cc1',                  0,'non','fragrance'),
    ('Lavender oil surrogate','CC(C)=CCCO',                     0,'non','terpene alcohol'),
    ('Nickel sulfate','[Ni+2].[O-]S(=O)(=O)[O-]',               1,'strong','metal contact allergy'),
    ('Cobalt chloride','[Co+2].[Cl-].[Cl-]',                    1,'moderate','metal'),
    ('Chrome sulfate','[Cr+3].[O-]S([O-])(=O)=O',               1,'strong','leather tanning'),
    ('Propylene glycol','CC(O)CO',                               0,'non','vehicle control'),
    ('SDS',       'CCCCCCCCCCCCOS(=O)(=O)[O-]',                0,'non','surfactant'),
    ('Ethanol',   'CCO',                                        0,'non','vehicle'),
    ('Penicillin G','CC1(C)SC2CC(NC(=O)Cc3ccccc3)(C(=O)O)C2(C)N1',1,'strong','beta-lactam'),
    ('Amoxicillin','CC1(C)SC2CC(NC(=O)C(N)c3ccc(O)cc3)(C(=O)O)C2(C)N1',1,'moderate','beta-lactam'),
    ('Chlorpromazine','CN(C)CCCN1c2ccccc2Sc2ccc(Cl)cc21',       1,'moderate','phenothiazine'),
    ('Benzocaine','CCOC(=O)c1ccc(N)cc1',                       1,'moderate','ester anaesthetic'),
    ('Methyldibromoglutaronitrile','N#CC(Br)CC(Br)C#N',          1,'extreme','preservative'),
    ('Dimethylfumarate','COC(=O)/C=C/C(=O)OC',                  1,'extreme','Michael acceptor'),
    ('Resorcinol','Oc1cccc(O)c1',                               0,'non','developer'),
    ('Hydroquinone','Oc1ccc(O)cc1',                              1,'moderate','oxidative'),
    ('Benzyl alcohol','OCc1ccccc1',                              0,'non','preservative'),
    ('Propyl paraben','CCCOC(=O)c1ccc(O)cc1',                   0,'non','preservative'),
    ('Thimerosal','CC[Hg]Sc1ccccc1C(=O)[O-].[Na+]',             1,'strong','organomercury'),
    ('Tea tree oil (terpinen-4-ol)','CC1CC(C(C)(C)O)CC(C)C1',   1,'moderate','oxidised terpene'),
    ('Balsam of Peru (benzyl cinnamate)','O=C(OCc1ccccc1)/C=C/c1ccccc1',1,'moderate','fragrance mix'),
    ('Kathon XL surrogate','ClCC(=O)Nc1ccccc1',                 1,'strong','chloroacetamide'),
    ('Propranolol','CC(C)NCC(O)COc1cccc2ccccc12',               0,'non','beta-blocker'),
    ('Salicylic acid','OC(=O)c1ccccc1O',                        0,'non','keratolytic'),
    ('4-Nitrobenzyl bromide','O=[N+]([O-])c1ccc(CBr)cc1',        1,'extreme','alkylating agent'),
]

df = pd.DataFrame(SKIN_DATA, columns=['name','smiles','sensitiser','potency','category'])
print(f'Dataset: {len(df)} chemicals | Sensitisers: {df.sensitiser.sum()} | Non: {(df.sensitiser==0).sum()}')

---
## Step 2 — 4 AOP-Anchored In-Vitro Assay Simulations

Each simulation models a real validated assay using mechanistic SMARTS and physicochemical rules.

In [ ]:
# ── DPRA (OECD TG 442C): covalent peptide binding ────────────────────────────
DPRA_SMARTS = [
    ('Michael_acceptor','C=CC(=O)'),('Aldehyde','[CX3H1](=O)'),
    ('Epoxide','C1OC1'),('Acyl_halide','C(=O)[Cl,Br,F]'),
    ('Nitro_reactive','[N+](=O)[O-]'),('Isocyanate','N=C=O'),
    ('Disulfide','SSC'),('SN2_halide','[CH2][Cl,Br,I]'),
]
np.random.seed(42)
def dpra_depletion(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if not mol: return 0.0, 0.0
    cys_hits = sum(1 for _,s in DPRA_SMARTS[:5]
                   if mol.HasSubstructMatch(Chem.MolFromSmarts(s)))
    lys_hits = sum(1 for _,s in DPRA_SMARTS[5:]
                   if mol.HasSubstructMatch(Chem.MolFromSmarts(s)))
    cys_dep  = min(100, cys_hits * 35 + np.random.uniform(0,15))
    lys_dep  = min(100, lys_hits * 25 + np.random.uniform(0,12))
    return round(cys_dep,1), round(lys_dep,1)

# ── KeratinoSens (OECD TG 442D): ARE-Nrf2 activation ─────────────────────────
def keratinosens_imax(smiles):
    """Simulate ARE-Nrf2 induction (Imax relative to positive control)."""
    mol = Chem.MolFromSmiles(smiles)
    if not mol: return 1.0
    n_elec = sum(1 for _,s in DPRA_SMARTS
                 if mol.HasSubstructMatch(Chem.MolFromSmarts(s)))
    logp   = Descriptors.MolLogP(mol)
    imax   = 1.0 + 1.5*n_elec + 0.3*max(0,logp-1) + np.random.uniform(-0.2,0.5)
    return round(max(1.0, imax), 2)

# ── h-CLAT (OECD TG 442E): CD54/CD86 on THP-1 cells ─────────────────────────
def hclat_cd_ratio(smiles):
    """Simulate h-CLAT CD54/CD86 expression ratio (>200% = positive)."""
    mol = Chem.MolFromSmiles(smiles)
    if not mol: return 100.0, 100.0
    n_elec = sum(1 for _,s in DPRA_SMARTS
                 if mol.HasSubstructMatch(Chem.MolFromSmarts(s)))
    mw     = Descriptors.MolWt(mol)
    cd54   = 100 + 80*n_elec + 20*np.random.randn()
    cd86   = 100 + 50*n_elec + 15*np.random.randn()
    return round(cd54,1), round(cd86,1)

# ── DEREK Nexus surrogate (QSAR structural rules) ─────────────────────────────
def derek_prediction(smiles):
    """Rule-based alert prediction (surrogate for Derek Nexus)."""
    mol = Chem.MolFromSmiles(smiles)
    if not mol: return 'Impossible', 'unknown'
    n = sum(1 for _,s in DPRA_SMARTS
            if mol.HasSubstructMatch(Chem.MolFromSmarts(s)))
    if   n >= 3: return 'Probable',  'extreme'
    elif n >= 2: return 'Plausible',  'strong'
    elif n == 1: return 'Equivocal', 'moderate'
    else:        return 'Improbable','non'

# ── Apply all four assays ─────────────────────────────────────────────────────
df[['dpra_cys','dpra_lys']] = pd.DataFrame(df['smiles'].apply(dpra_depletion).tolist(),index=df.index)
df['ks_imax']               = df['smiles'].apply(keratinosens_imax)
df[['hclat_cd54','hclat_cd86']] = pd.DataFrame(df['smiles'].apply(hclat_cd_ratio).tolist(),index=df.index)
df[['derek_call','derek_potency']] = pd.DataFrame(df['smiles'].apply(derek_prediction).tolist(),index=df.index)

# Binary calls per assay
df['dpra_pos']  = ((df['dpra_cys'] >= 13.89) | (df['dpra_lys'] >= 22.62)).astype(int)
df['ks_pos']    = (df['ks_imax'] >= 1.5).astype(int)
df['hclat_pos'] = ((df['hclat_cd54'] >= 200) | (df['hclat_cd86'] >= 150)).astype(int)
df['derek_pos'] = df['derek_call'].isin(['Probable','Plausible']).astype(int)

print('AOP assay simulations complete.')
print(df[['name','sensitiser','dpra_pos','ks_pos','hclat_pos','derek_pos']].to_string(index=False))

---
## Step 3 — OECD TG 497 Defined Approaches

In [ ]:
# ── DA 1: 2o3 (any 2 of 3 in-vitro positives = sensitiser call) ──────────────
df['da_2o3'] = ((df['dpra_pos'] + df['ks_pos'] + df['hclat_pos']) >= 2).astype(int)

# ── DA 2: ITS-3 (DPRA + KS + h-CLAT + QSAR score) ───────────────────────────
def its3_score(row):
    """Integrated Testing Strategy 3 score (0-4)."""
    return row['dpra_pos'] + row['ks_pos'] + row['hclat_pos'] + row['derek_pos']

df['its3_score'] = df.apply(its3_score, axis=1)
df['da_its3']    = (df['its3_score'] >= 2).astype(int)

# ── DA 3: RF QSAR model on all features ──────────────────────────────────────
def featurise(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if not mol: return np.zeros(2048+6)
    fp  = AllChem.GetMorganFingerprintAsBitVect(mol,2,2048)
    arr = np.zeros((2048,))
    from rdkit import DataStructs; DataStructs.ConvertToNumpyArray(fp,arr)
    phys = np.array([Descriptors.MolWt(mol),Descriptors.MolLogP(mol),
                     Descriptors.TPSA(mol),Descriptors.NumHDonors(mol),
                     Descriptors.NumHAcceptors(mol),Descriptors.HeavyAtomCount(mol)])
    return np.concatenate([arr,phys])

X = np.hstack([
    np.vstack(df['smiles'].apply(featurise).values),
    df[['dpra_cys','dpra_lys','ks_imax','hclat_cd54','hclat_cd86','its3_score']].values
])
y = df['sensitiser'].values

rf = RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=42)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
p  = cross_val_predict(rf, X, y, cv=cv, method='predict_proba')[:,1]
df['rf_prob'] = p
df['da_rf']   = (p >= 0.5).astype(int)

# ── Performance comparison ────────────────────────────────────────────────────
print('Performance comparison (GPMT replacement):')
print(f'{'Method':20s} {'AUC':>6} {'MCC':>6} {'Acc':>6}')
for name, pred, is_binary in [
    ('2o3 DA',     df['da_2o3'], True),
    ('ITS-3 DA',   df['da_its3'],True),
    ('RF QSAR',    df['da_rf'],  True),
    ('DPRA alone', df['dpra_pos'],True),
    ('KS alone',   df['ks_pos'], True),
    ('h-CLAT alone',df['hclat_pos'],True),
]:
    from sklearn.metrics import accuracy_score
    auc_i = roc_auc_score(y, pred)
    mcc_i = matthews_corrcoef(y, pred)
    acc_i = accuracy_score(y, pred)
    print(f'{name:20s} {auc_i:>6.3f} {mcc_i:>6.3f} {acc_i:>6.3f}')

---
## Step 4 — Agentic IATA Classifier (GPT-4o)

In [ ]:
client = OpenAI(api_key=os.getenv('OPENAI_API_KEY',''))

def tool_dpra(name): row=df[df['name']==name].iloc[0]; return {'compound':name,'cys_dep':row['dpra_cys'],'lys_dep':row['dpra_lys'],'positive':bool(row['dpra_pos'])}
def tool_ks(name):   row=df[df['name']==name].iloc[0]; return {'compound':name,'imax':row['ks_imax'],'positive':bool(row['ks_pos'])}
def tool_hclat(name):row=df[df['name']==name].iloc[0]; return {'compound':name,'cd54':row['hclat_cd54'],'cd86':row['hclat_cd86'],'positive':bool(row['hclat_pos'])}
def tool_derek(name):row=df[df['name']==name].iloc[0]; return {'compound':name,'call':row['derek_call'],'potency':row['derek_potency'],'positive':bool(row['derek_pos'])}
def tool_da_classify(name):row=df[df['name']==name].iloc[0]; return {'compound':name,'2o3':int(row['da_2o3']),'its3':int(row['da_its3']),'rf_prob':round(row['rf_prob'],3),'its3_score':int(row['its3_score'])}

TOOL_REGISTRY={'dpra':tool_dpra,'keratinosens':tool_ks,'hclat':tool_hclat,'derek':tool_derek,'da_classify':tool_da_classify}
TOOLS=[{'type':'function','function':{'name':k,'description':f'{k} skin sensitisation assay.',
         'parameters':{'type':'object','properties':{'name':{'type':'string'}},'required':['name']}}} for k in TOOL_REGISTRY]

def run_skin_agent(chem_name):
    if not AGENT_OK:
        row=df[df['name']==chem_name].iloc[0]
        return f'[Demo] {chem_name}: 2o3={row["da_2o3"]}, ITS3={row["da_its3"]}, true={row["sensitiser"]}'
    msgs=[{'role':'system','content':'Skin sensitisation IATA agent. Use all 4 assay tools then da_classify.'},
          {'role':'user','content':f'Classify {chem_name} for skin sensitisation under OECD TG 497.'}]
    for _ in range(8):
        r=client.chat.completions.create(model='gpt-4o',messages=msgs,tools=TOOLS,tool_choice='auto')
        c=r.choices[0]; msgs.append(c.message)
        if c.finish_reason=='stop': return c.message.content
        for tc in c.message.tool_calls:
            res=TOOL_REGISTRY[tc.function.name](**json.loads(tc.function.arguments))
            msgs.append({'role':'tool','tool_call_id':tc.id,'content':json.dumps(res)})
    return 'max iter'

for c in ['Cinnamaldehyde','Eugenol','Linalool','Propylene glycol']:
    print(f'--- {c} ---'); print(run_skin_agent(c)); print()

---
## Step 5 — Skin Sensitisation Dashboard

In [ ]:
fig = plt.figure(figsize=(20,13))
gs  = gridspec.GridSpec(2,3,hspace=0.45,wspace=0.35)

# P1: AOP heatmap (assay positives)
ax1=fig.add_subplot(gs[0,0:2])
aop_cols=['dpra_pos','ks_pos','hclat_pos','derek_pos','da_2o3','da_its3']
aop_labels=['DPRA','KeratinoSens','h-CLAT','DEREK','2o3 DA','ITS-3 DA']
sort_df=df.sort_values('its3_score',ascending=False)
heat=sort_df[aop_cols].values
im=ax1.imshow(heat.T,cmap='RdYlGn_r',aspect='auto',vmin=0,vmax=1)
ax1.set_yticks(range(6)); ax1.set_yticklabels(aop_labels,fontsize=9)
ax1.set_xticks(range(len(sort_df)))
ax1.set_xticklabels(sort_df['name'],rotation=60,ha='right',fontsize=7)
# Mark true positives
for xi,(_,row) in enumerate(sort_df.iterrows()):
    if row['sensitiser']:
        ax1.axvline(xi,c='blue',lw=0.4,alpha=0.25)
ax1.set_title('AOP-Anchored Assay Results (sorted by ITS-3 score)\n'
              'Blue = true sensitiser',fontweight='bold')
plt.colorbar(im,ax=ax1,shrink=0.8,label='Result')

# P2: Confusion matrix (ITS-3)
ax2=fig.add_subplot(gs[0,2])
cm=confusion_matrix(y,df['da_its3'])
im2=ax2.imshow(cm,cmap='Blues')
for i in range(2):
    for j in range(2):
        ax2.text(j,i,cm[i,j],ha='center',va='center',fontweight='bold',fontsize=14,
                 color='white' if cm[i,j]>cm.max()/2 else 'black')
ax2.set_xticks([0,1]); ax2.set_yticks([0,1])
ax2.set_xticklabels(['Pred NS','Pred S']); ax2.set_yticklabels(['True NS','True S'])
ax2.set_title(f'ITS-3 DA Confusion Matrix\nAUC={roc_auc_score(y,df["da_its3"]):.3f}',fontweight='bold')
plt.colorbar(im2,ax=ax2,shrink=0.8)

# P3: DPRA depletion scatter
ax3=fig.add_subplot(gs[1,0])
cols=[('#E74C3C' if s else '#27AE60') for s in df['sensitiser']]
ax3.scatter(df['dpra_cys'],df['dpra_lys'],c=cols,s=70,edgecolors='k',lw=0.5,alpha=0.9)
ax3.axvline(13.89,c='r',ls='--',lw=1.5,label='Cys threshold 13.89%')
ax3.axhline(22.62,c='orange',ls='--',lw=1.5,label='Lys threshold 22.62%')
ax3.set_xlabel('DPRA Cys depletion %'); ax3.set_ylabel('DPRA Lys depletion %')
ax3.set_title('DPRA Cys vs Lys Depletion\n(Red=sensitiser)',fontweight='bold')
ax3.legend(fontsize=8); ax3.grid(True,alpha=0.3)

# P4: ITS-3 score distribution
ax4=fig.add_subplot(gs[1,1])
for val,col,label in [(1,'#E74C3C','Sensitiser'),(0,'#27AE60','Non-sensitiser')]:
    ax4.hist(df[df['sensitiser']==val]['its3_score'],bins=np.arange(-0.5,5.5,1),
             color=col,alpha=0.65,label=label,edgecolor='white')
ax4.axvline(2,c='k',ls='--',lw=2,label='ITS-3 threshold=2')
ax4.set_xlabel('ITS-3 score (0-4)'); ax4.set_ylabel('Count')
ax4.set_title('ITS-3 Score Distribution',fontweight='bold')
ax4.legend(); ax4.grid(True,alpha=0.3,axis='y')

# P5: Animal replacement comparison
ax5=fig.add_subplot(gs[1,2])
from sklearn.metrics import accuracy_score
methods=['DPRA\nalone','KeratinoSens\nalone','h-CLAT\nalone','2o3\nDA','ITS-3\nDA','RF\nQSAR']
preds=[df['dpra_pos'],df['ks_pos'],df['hclat_pos'],df['da_2o3'],df['da_its3'],df['da_rf']]
accs=[accuracy_score(y,p)*100 for p in preds]
cols2=['#3498DB','#9B59B6','#E67E22','#1ABC9C','#E74C3C','#2ECC71']
bars=ax5.bar(methods,accs,color=cols2,alpha=0.85,edgecolor='white',width=0.55)
ax5.axhline(75,c='k',ls='--',lw=2,alpha=0.6,label='GPMT accuracy ~75%')
for bar,v in zip(bars,accs):
    ax5.text(bar.get_x()+bar.get_width()/2,bar.get_height()+0.5,f'{v:.0f}%',
             ha='center',va='bottom',fontweight='bold',fontsize=9)
ax5.set_ylabel('Accuracy (%)'); ax5.set_ylim(0,105)
ax5.set_title('NAM vs GPMT Accuracy',fontweight='bold')
ax5.legend(fontsize=8); ax5.grid(True,alpha=0.3,axis='y')

plt.suptitle('NAM Tutorial 17 — Skin Sensitisation Defined Approach\n'
             'OECD TG 497 DA vs Guinea Pig Maximisation Test (n=40)',fontsize=14,fontweight='bold')
plt.savefig('skin_output/nam17_skin_dashboard.png',dpi=130,bbox_inches='tight')
plt.show()
print('Saved: skin_output/nam17_skin_dashboard.png')